# EmoBench Synthetic Data Generation Pipeline
## Multi-Agent Dialogue System (MADS) for Emotional Understanding & Application

This notebook implements a hybrid multi-agent + single-agent pipeline to generate EmoBench-style EU (Emotional Understanding) and EA (Emotional Application) examples.

**Pipeline Overview:**
1. **Background Generator Agent**: Creates elaborate backgrounds for patients
2. **Client Agent**: Absorbs persona + background, engages in therapy-like dialogue
3. **Therapist Agent**: Structured therapeutic dialogue with probing questions
4. **Supervisor/Orchestrator**: Monitors dialogue quality and stops when ready
5. **Single-Agent Extractor**: Distills dialogues into clean EU/EA items



In [1]:
# Install required packages
%pip install vllm openai jsonlines tqdm python-dotenv requests -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/1

# 🎯 Improved EmoBench Generation Pipeline

## Key Improvements

### 1. **Category-Aware Upsampling** 📊
- Generates 3x more samples for underperforming categories:
  - EU: `personal_beliefs_and_experiences` (0.268), `perspective_taking` (0.269)
  - EA: `Social-Others` (0.54), `Personal-Others` (0.56)

### 2. **Quality Filtering** ✅
- Automatically rejects poor generations:
  - No explicit emotion words in EU scenarios
  - Subject must be mentioned in EA scenarios
  - Minimum length requirements
  - Correct number of choices

### 3. **Increased Volume** 🚀
- 3000 dialogues × 4 items each = ~8K-10K items per category
- With upsampling: weak categories get 3x more coverage

### 4. **Category Tracking** 📈
- Real-time monitoring of category distribution
- Detailed report at end of generation

## Expected Results

### Before (Current Fine-Tuning)
- EU Overall: **0.33** (minimal improvement)
- EA Overall: **0.615** (worse than baseline 0.68) ❌

### After (New Pipeline + Fine-Tuning)
- EU Overall: **0.40+** (30% improvement target)
- EA Overall: **0.68+** (match/beat baseline)
- All categories improve ✅

## Usage

1. **Start vLLM Server** (Cell 14)
2. **Run Generation** (Cell 35)
3. **Check Distribution** (automatic report at end)
4. **Fine-Tune** (see `finetune_gpt_oss_20b.ipynb`)
5. **Evaluate** (compare against baseline)

See `IMPROVEMENT_SUMMARY.md` for full details.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import jsonlines
import random
import os
import time
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

print("✓ Libraries imported")
print("✓ Using vLLM with OpenAI-compatible API")



✓ Libraries imported
✓ Using vLLM with OpenAI-compatible API


## Persona-Theme Compatibility Checker

Before generating dialogues, we check if the persona and theme are compatible to avoid incoherent combinations.


In [4]:
# Persona-Theme Compatibility Checker
# Uses keyword matching and semantic similarity to ensure coherent combinations

def check_persona_theme_compatibility(persona_text: str, theme: Dict) -> Tuple[bool, float, str]:
    """
    Check if persona and theme are compatible.
    Returns: (is_compatible, confidence_score, reason)
    """
    persona_lower = persona_text.lower()
    theme_name = theme.get("theme", "").lower()
    theme_desc = theme.get("description", "").lower()
    theme_keywords = [kw.lower() for kw in theme.get("keywords", [])]
    age_relevance = theme.get("age_relevance", "all")

    # Age-based compatibility rules
    age_incompatibilities = {
        "student": ["retirement", "elderly", "senior", "midlife", "empty nest", "aging parent"],
        "adult": ["academic pressure", "choosing a major", "student debt", "graduate school"],
        "adult_elderly": ["academic", "student", "university", "college", "school", "exam"],
        "elderly": ["academic", "student", "university", "career transition", "job", "workplace"]
    }

    # Check for obvious age incompatibilities
    if age_relevance == "student":
        # Persona should contain student-related terms
        student_indicators = ["student", "school", "university", "college", "academic", "study", "learn"]
        if not any(indicator in persona_lower for indicator in student_indicators):
            # Check if it's clearly NOT a student
            non_student_indicators = ["retired", "elderly", "senior", "professional", "manager", "director", "executive"]
            if any(indicator in persona_lower for indicator in non_student_indicators):
                return False, 0.0, f"Persona appears non-student but theme requires student context"

    if age_relevance == "adult_elderly" or age_relevance == "elderly":
        # Persona should indicate adult/elderly
        adult_indicators = ["retired", "elderly", "senior", "professional", "manager", "director", "executive", "parent", "adult"]
        student_indicators = ["student", "school", "university", "college", "academic"]
        if any(indicator in persona_lower for indicator in student_indicators):
            if not any(indicator in persona_lower for indicator in adult_indicators):
                return False, 0.0, f"Persona appears student but theme requires adult/elderly context"

    # Keyword-based compatibility (positive signals)
    keyword_matches = sum(1 for kw in theme_keywords if kw in persona_lower)
    keyword_score = keyword_matches / len(theme_keywords) if theme_keywords else 0

    # Category-based compatibility
    category_keywords = {
        "Professional": ["professional", "work", "job", "career", "business", "manager", "director", "executive", "employee"],
        "Education": ["student", "school", "university", "college", "academic", "teacher", "professor", "education"],
        "Family": ["parent", "family", "child", "children", "sibling", "relative", "parenting"],
        "Relationships": ["relationship", "partner", "spouse", "dating", "romantic", "friend"],
        "Health": ["health", "medical", "doctor", "patient", "treatment", "illness"],
        "Financial": ["financial", "money", "debt", "income", "investment", "wealth"]
    }

    theme_category = theme.get("category", "")
    category_score = 0.0
    if theme_category in category_keywords:
        category_kws = category_keywords[theme_category]
        category_matches = sum(1 for kw in category_kws if kw in persona_lower)
        category_score = category_matches / len(category_kws) if category_kws else 0

    # Overall compatibility score
    compatibility_score = (keyword_score * 0.6 + category_score * 0.4)

    # Threshold for compatibility (adjust as needed)
    COMPATIBILITY_THRESHOLD = 0.1  # Low threshold - only filter out very obvious mismatches

    is_compatible = compatibility_score >= COMPATIBILITY_THRESHOLD

    reason = f"Score: {compatibility_score:.2f} (keyword: {keyword_score:.2f}, category: {category_score:.2f})"

    return is_compatible, compatibility_score, reason


def find_compatible_themes(persona_text: str, themes: List[Dict], max_attempts: int = 10) -> Optional[Dict]:
    """
    Find a compatible theme for the persona.
    Tries up to max_attempts to find a compatible theme.
    """
    compatible_themes = []
    all_scores = []

    for theme in themes:
        is_compat, score, reason = check_persona_theme_compatibility(persona_text, theme)
        all_scores.append((theme, score, is_compat))
        if is_compat:
            compatible_themes.append((theme, score))

    if compatible_themes:
        # Sort by score and pick from top 50% (adds some randomness while preferring good matches)
        compatible_themes.sort(key=lambda x: x[1], reverse=True)
        top_n = max(1, len(compatible_themes) // 2)
        return random.choice(compatible_themes[:top_n])[0]
    else:
        # If no compatible themes found, return highest scoring one anyway
        # (better than failing completely)
        all_scores.sort(key=lambda x: x[1], reverse=True)
        if all_scores:
            return all_scores[0][0]
        return random.choice(themes)


print("✓ Persona-Theme compatibility checker defined")


✓ Persona-Theme compatibility checker defined


In [ ]:
# This cell was moved to after themes are loaded (see Cell 13)
# Compatibility checker functions are defined in Cell 4


## File Access Setup

Choose one of the following methods to access your files:
1. **Mount Google Drive** (recommended if files are already in Drive)
2. **Upload files directly** (if files are on your local computer)


In [5]:
# OPTION 1: Mount Google Drive (Recommended)
# If your files are already in Google Drive, use this method



# Set the path to your project folder in Google Drive
# Example: if your project is in "My Drive/FINAL PROJECT", set:
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/685_Project_New/"  # Adjust this path!

# Verify the path exists
if os.path.exists(DRIVE_PROJECT_PATH):
    print(f"✓ Google Drive mounted successfully!")
    print(f"✓ Project path: {DRIVE_PROJECT_PATH}")
    # List files in the directory
    files = os.listdir(DRIVE_PROJECT_PATH)
    print(f"✓ Files found: {files}")
else:
    print(f"⚠ Path not found: {DRIVE_PROJECT_PATH}")
    print("Please adjust DRIVE_PROJECT_PATH to match your Google Drive folder structure")


✓ Google Drive mounted successfully!
✓ Project path: /content/drive/MyDrive/685_Project_New/
✓ Files found: ['emobench_data', 'input_data', 'generated_data', 'fine-tuned_model']


In [6]:
# OPTION 2: Upload files directly from your computer
# Use this if you want to upload files directly without using Google Drive

from google.colab import files
import os

# Create upload directory
INPUT_DIR = "/content/drive/MyDrive/685_Project_New/input_data/mads/"
# os.makedirs(INPUT_DIR, exist_ok=True)

# print("Upload your files now:")
# print("1. persona.jsonl")
# print("2. themes.jsonl")
# print("\nClick 'Choose Files' button below and select your files...")

# # Upload files
# uploaded = files.upload()

# # Move uploaded files to the upload directory
# for filename in uploaded.keys():
#     os.rename(filename, os.path.join(UPLOAD_DIR, filename))
#     print(f"✓ Uploaded: {filename}")

# # Set project path to uploaded files directory
# UPLOAD_PROJECT_PATH = UPLOAD_DIR

# print(f"\n✓ Files uploaded to: {UPLOAD_PROJECT_PATH}")
# print(f"✓ Files available: {os.listdir(UPLOAD_PROJECT_PATH)}")


In [7]:
# Set the project path based on which method you used above
# Uncomment ONE of the following:

# If you used Google Drive (Option 1):
try:
    PROJECT_PATH = INPUT_DIR
except NameError:
    PROJECT_PATH = None

# If you used direct upload (Option 2), uncomment this instead:
# try:
#     PROJECT_PATH = UPLOAD_PROJECT_PATH
# except NameError:
#     PROJECT_PATH = None

# If you want to use files in the current Colab directory:
# PROJECT_PATH = "/content"

# If PROJECT_PATH is not set, default to current directory
if PROJECT_PATH is None:
    PROJECT_PATH = "/content"
    print("⚠ Using default path: /content")
    print("Please run either Option 1 (Google Drive) or Option 2 (Upload) cells above first!")

print(f"Using project path: {PROJECT_PATH}")

# Verify required files exist
required_files = ["persona.jsonl", "themes.jsonl"]
missing_files = []

for file in required_files:
    file_path = os.path.join(PROJECT_PATH, file)
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
        print(f"✓ Found: {file} ({file_size:.2f} MB)")
    else:
        print(f"✗ Missing: {file}")
        missing_files.append(file)

if missing_files:
    print(f"\n⚠ Warning: Missing files: {missing_files}")
    print("Please ensure all required files are available in the project path.")
else:
    print("\n✓ All required files found!")


Using project path: /content/drive/MyDrive/685_Project_New/input_data/mads/
✓ Found: persona.jsonl (20.62 MB)
✓ Found: themes.jsonl (0.02 MB)

✓ All required files found!


**Important:** Before loading personas, make sure you've run one of the file access cells above and set `PROJECT_PATH`.

If you see an error about `PROJECT_PATH` not being defined, go back and run either:
- Cell 4 (Google Drive mount) + Cell 6 (Set PROJECT_PATH), OR  
- Cell 5 (File upload) + Cell 6 (Set PROJECT_PATH)


In [8]:
# Quick fix: Update file paths to use PROJECT_PATH
# Run this cell if the persona loading cell below fails

# Update the load_jsonl calls to use PROJECT_PATH
import os

# Make sure PROJECT_PATH is set
if 'PROJECT_PATH' not in globals():
    print("⚠ PROJECT_PATH not set! Please run the file access setup cells above first.")
    PROJECT_PATH = "/content"  # Fallback
    print(f"Using fallback path: {PROJECT_PATH}")

# These will be used in the next cell
PERSONA_FILE = os.path.join(PROJECT_PATH, "persona.jsonl")
THEMES_FILE = os.path.join(PROJECT_PATH, "themes.jsonl")

print(f"Persona file path: {PERSONA_FILE}")
print(f"Themes file path: {THEMES_FILE}")
print(f"Persona file exists: {os.path.exists(PERSONA_FILE)}")
print(f"Themes file exists: {os.path.exists(THEMES_FILE)}")


Persona file path: /content/drive/MyDrive/685_Project_New/input_data/mads/persona.jsonl
Themes file path: /content/drive/MyDrive/685_Project_New/input_data/mads/themes.jsonl
Persona file exists: True
Themes file exists: True


In [9]:
# Load personas and themes
def load_jsonl(filepath: str) -> List[Dict]:
    """Load JSONL file into list of dictionaries"""
    data = []
    with jsonlines.open(filepath) as reader:
        for obj in reader:
            data.append(obj)
    return data

# Load data - uses PROJECT_PATH from file access setup cells above
# If PERSONA_FILE and THEMES_FILE are set (from previous cell), use those
# Otherwise, try to construct paths from PROJECT_PATH
try:
    persona_file = PERSONA_FILE
    themes_file = THEMES_FILE
except NameError:
    # Fallback: construct paths from PROJECT_PATH
    if 'PROJECT_PATH' in globals():
        persona_file = os.path.join(PROJECT_PATH, "persona.jsonl")
        themes_file = os.path.join(PROJECT_PATH, "themes.jsonl")
    else:
        # Last resort: current directory
        persona_file = "./persona.jsonl"
        themes_file = "./themes.jsonl"
        print("⚠ Warning: Using current directory. Make sure files are uploaded!")

print(f"Loading personas from: {persona_file}")
personas = load_jsonl(persona_file)

print(f"Loading themes from: {themes_file}")
themes = load_jsonl(themes_file)

print(f"\n✓ Loaded {len(personas)} personas")
print(f"✓ Loaded {len(themes)} themes")

# Sample to verify
print(f"\nSample persona: {personas[0]}")
print(f"Sample theme: {themes[0]}")



Loading personas from: /content/drive/MyDrive/685_Project_New/input_data/mads/persona.jsonl
Loading themes from: /content/drive/MyDrive/685_Project_New/input_data/mads/themes.jsonl

✓ Loaded 200000 personas
✓ Loaded 92 themes

Sample persona: {'persona': "A Political Analyst specialized in El Salvador's political landscape."}
Sample theme: {'theme': 'Career Transition', 'category': 'Professional', 'description': 'Major career changes, job loss, promotion, industry shifts', 'age_relevance': 'adult', 'keywords': ['career', 'job', 'work', 'professional', 'employment']}


In [10]:
# Test the compatibility checker
# Now that themes are loaded, we can test the compatibility functions

print("Testing persona-theme compatibility...\n")

# Test cases
test_cases = [
    ("A 60-year-old retired teacher", "Academic Pressure"),  # Should be incompatible
    ("A university student studying computer science", "Academic Pressure"),  # Should be compatible
    ("A young professional starting their career", "Retirement Planning"),  # Should be incompatible
    ("A parent of three children", "Parenting Challenges"),  # Should be compatible
    ("A retired executive", "Career Transition"),  # Should be compatible (career-related)
]

for persona_text, theme_name in test_cases:
    # Find the theme
    theme = next((t for t in themes if t.get("theme") == theme_name), None)
    if theme:
        is_compat, score, reason = check_persona_theme_compatibility(persona_text, theme)
        status = "✓ Compatible" if is_compat else "✗ Incompatible"
        print(f"{status}: '{persona_text}' + '{theme_name}'")
        print(f"  {reason}\n")
    else:
        print(f"Theme '{theme_name}' not found\n")


Testing persona-theme compatibility...

✗ Incompatible: 'A 60-year-old retired teacher' + 'Academic Pressure'
  Persona appears non-student but theme requires student context

✓ Compatible: 'A university student studying computer science' + 'Academic Pressure'
  Score: 0.36 (keyword: 0.43, category: 0.25)

✗ Incompatible: 'A young professional starting their career' + 'Retirement Planning'
  Score: 0.00 (keyword: 0.00, category: 0.00)

✓ Compatible: 'A parent of three children' + 'Parenting Challenges'
  Score: 0.53 (keyword: 0.60, category: 0.43)

✗ Incompatible: 'A retired executive' + 'Career Transition'
  Score: 0.04 (keyword: 0.00, category: 0.11)



In [13]:
!nohup python -m vllm.entrypoints.openai.api_server \
    --model openai/gpt-oss-20b \
    --host 0.0.0.0 \
    --port 8000 > server.log 2>&1 &


In [14]:
# Model Configuration - vLLM with GPT-OSS-20B
# First, start the vLLM server in a separate terminal/cell:
# !nohup python -m vllm.entrypoints.openai.api_server \
#     --model openai/gpt-oss-20b \
#     --host 0.0.0.0 \
#     --port 8000 > server.log 2>&1 &
#
# Wait for the server to start (check server.log or wait ~30 seconds)
# The server will be available at http://localhost:8000/v1

MODEL_NAME = "openai/gpt-oss-20b"
VLLM_BASE_URL = "http://localhost:8000/v1"  # vLLM OpenAI-compatible API endpoint

print(f"✓ Model: {MODEL_NAME}")
print(f"✓ vLLM Server URL: {VLLM_BASE_URL}")

# Initialize OpenAI client pointing to vLLM server
client = OpenAI(
    base_url=VLLM_BASE_URL,
    api_key="EMPTY"  # vLLM doesn't require a real API key
)

print("✓ OpenAI client initialized for vLLM")

# LLM Call Helper Function
def call_llm(
    prompt: str,
    model_name: Optional[str] = None,  # Kept for compatibility, not used
    temperature: float = 0.7,
    max_tokens: int = 2000,
    system_prompt: Optional[str] = None
) -> str:
    """
    Call GPT-OSS-20B via vLLM using OpenAI-compatible API.
    GPT-OSS-20B is a base model, so we use completions API (plain text) instead of chat.
    """
    # Prepare plain text prompt (GPT-OSS-20B is a base model, not chat-tuned)
    if system_prompt:
        full_prompt = f"{system_prompt}\n\n{prompt}"
    else:
        full_prompt = prompt

    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Use completions API (plain text) - GPT-OSS-20B works better with this
            response = client.completions.create(
                model=MODEL_NAME,
                prompt=full_prompt,
                temperature=temperature,
                max_tokens=max_tokens,
                stop=None,  # Let model decide when to stop
            )

            # Extract the response text
            if response.choices and len(response.choices) > 0:
                content = response.choices[0].text
                if content is not None and content.strip():
                    return content.strip()

            # If empty, try chat completions as fallback (some vLLM setups support it)
            if attempt < max_retries - 1:
                try:
                    messages = []
                    if system_prompt:
                        messages.append({"role": "system", "content": system_prompt})
                    messages.append({"role": "user", "content": prompt})

                    chat_response = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=messages,
                        temperature=temperature,
                        max_tokens=max_tokens,
                    )
                    if chat_response.choices and len(chat_response.choices) > 0:
                        content = chat_response.choices[0].message.content
                        if content is not None and content.strip():
                            return content.strip()
                except:
                    pass  # Continue to next retry

        except Exception as e:
            if attempt == max_retries - 1:
                # Last attempt failed
                print(f"Error calling vLLM after {max_retries} attempts: {e}")
                return ""
            time.sleep(0.5)  # Brief wait before retry
            continue

    # If we get here, all attempts failed
    print(f"Warning: vLLM returned None/empty content after {max_retries} attempts")
    return ""

print("✓ LLM helper function defined")



✓ Model: openai/gpt-oss-20b
✓ vLLM Server URL: http://localhost:8000/v1
✓ OpenAI client initialized for vLLM
✓ LLM helper function defined


In [17]:
# Test vLLM server connection and response
# Run this cell to debug if you're getting None responses

print("Testing vLLM server...")

# Test 1: Health check
try:
    import requests
    health = requests.get("http://localhost:8000/health", timeout=5)
    print(f"✓ Health check: {health.status_code}")
except Exception as e:
    print(f"✗ Health check failed: {e}")

# Test 2: Simple chat completion
try:
    test_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "Say hello"}],
        temperature=0.7,
        max_tokens=50
    )
    print(f"\nChat Completions Response:")
    print(f"  Response type: {type(test_response)}")
    print(f"  Has choices: {hasattr(test_response, 'choices')}")
    if hasattr(test_response, 'choices'):
        print(f"  Number of choices: {len(test_response.choices) if test_response.choices else 0}")
        if test_response.choices and len(test_response.choices) > 0:
            choice = test_response.choices[0]
            print(f"  Choice type: {type(choice)}")
            print(f"  Has message: {hasattr(choice, 'message')}")
            if hasattr(choice, 'message'):
                msg = choice.message
                print(f"  Message type: {type(msg)}")
                print(f"  Has content: {hasattr(msg, 'content')}")
                print(f"  Content: {msg.content}")
                print(f"  Content type: {type(msg.content)}")
except Exception as e:
    print(f"✗ Chat completions test failed: {e}")
    import traceback
    traceback.print_exc()

# Test 3: Completions API (plain text)
try:
    completion_response = client.completions.create(
        model=MODEL_NAME,
        prompt="Say hello",
        temperature=0.7,
        max_tokens=50
    )
    print(f"\nCompletions API Response:")
    print(f"  Response type: {type(completion_response)}")
    if hasattr(completion_response, 'choices') and completion_response.choices:
        print(f"  Text: {completion_response.choices[0].text}")
except Exception as e:
    print(f"✗ Completions API test failed: {e}")
    import traceback
    traceback.print_exc()

print("\nIf both tests return None/empty, check server.log for errors.")


Testing vLLM server...
✓ Health check: 200

Chat Completions Response:
  Response type: <class 'openai.types.chat.chat_completion.ChatCompletion'>
  Has choices: True
  Number of choices: 1
  Choice type: <class 'openai.types.chat.chat_completion.Choice'>
  Has message: True
  Message type: <class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
  Has content: True
  Content: Hello! 👋
  Content type: <class 'str'>

Completions API Response:
  Response type: <class 'openai.types.completion.Completion'>
  Text:  to our new Customer Experience Champion. She is a brilliant and innovative manager, and we are delighted that she has joined our team.

We are looking forward to learning from her and every new experience let's all start the big change

Sure! Here's a revised

If both tests return None/empty, check server.log for errors.


In [18]:
# Extract MULTIPLE EU and EA items from a single dialogue
# This allows us to generate multiple examples from one therapist-patient conversation

def extract_multiple_eu_items(
    dialogue: List[Tuple[str, str]],
    metadata: Dict,
    num_items: int = 3,
    max_retries: int = 3
) -> List[Dict]:
    """
    Extract MULTIPLE Emotional Understanding MCQs from a single dialogue.
    Returns a list of EU items (can be empty if extraction fails).
    """
    dialogue_str = "\n".join([
        f"{role.upper()}: {msg}" for role, msg in dialogue
    ])

    # Truncate dialogue if too long
    if len(dialogue_str) > 3000:
        dialogue_str = dialogue_str[:3000] + "..."

    prompt = f"""You are an expert at extracting emotional understanding items from therapy dialogues.

DIALOGUE:
{dialogue_str}

**Task**: Extract {num_items} DIFFERENT Emotional Understanding MCQs from this dialogue. Each MCQ should focus on a DIFFERENT emotional moment, scenario, or situation mentioned in the dialogue. You MUST output a JSON array with {num_items} items, nothing else.

**IMPORTANT**:
- Each item should focus on a DIFFERENT part of the dialogue or a DIFFERENT emotional situation
- Emotions can be SINGLE emotions OR combinations (use & to combine)
- Each item should be unique and distinct from the others

Required JSON format (array of objects):
[
    {{
        "qid": "1",
        "language": "en",
        "coarse_category": "complex_emotions",
        "finegrained_category": "mixture_of_emotions",
        "scenario": "A clear scenario WITHOUT explicit emotion words (no 'sad', 'angry', etc.)",
        "subject": "I" or character name,
        "emotion_choices": [
            "Single emotion OR Emotion1 & Emotion2",
            "Single emotion OR Emotion2 & Emotion3",
            "Single emotion OR Emotion3 & Emotion4",
            "Single emotion OR Emotion4 & Emotion5",
            "Single emotion OR Emotion5 & Emotion6"
        ],
        "emotion_label": "The correct emotion (single or combination)",
        "cause_choices": [
            "Single cause OR Cause1 & Cause2",
            "Single cause OR Cause2 & Cause3",
            "Single cause OR Cause3 & Cause4",
            "Single cause OR Cause4 & Cause5",
            "Single cause OR Cause5 & Cause6"
        ],
        "cause_label": "The correct cause (single or combination)"
    }},
    {{
        "qid": "2",
        ...
    }},
    ...
]


**Examples:**

Example 1 (Single Emotion):
{{
    "qid": "2",
    "language": "en",
    "coarse_category": "complex_emotions",
    "finegrained_category": "emotion_transition",
    "scenario": "Simon, an amateur painter, had been working on a portrait of his deceased pet for over a week. After he finally completed it, he stepped back and immediately noticed that he had painted the wrong color for his pet's fur. Just then, his wife saw it and exclaimed, \"That's exactly how I remember him!\"",
    "subject": "Simon",
    "emotion_choices": [
        "Delight",
        "Anger",
        "Embarrassment",
        "Hopeless",
        "Pride",
        "Disappointment"
    ],
    "emotion_label": "Delight",
    "cause_choices": [
        "He thought he had painted the perfect portrait",
        "His wife appreciated his effort and liked his portrait",
        "He made a mistake in a painting he cared about",
        "His wife appreciated his effort despite noticing the mistake"
    ],
    "cause_label": "His wife appreciated his effort and liked his portrait"
}}

Example 2 (Combination):
{{
    "qid": "21",
    "language": "en",
    "coarse_category": "complex_emotions",
    "finegrained_category": "mixture_of_emotions",
    "scenario": "I applied to eight different universities to pursue my Bachelor's degree. Each decision letter came in differing dates. I waited until I received all the decision letters and opened them simultaneously. After opening the letters, I found that five of the universities, including my dream university, rejected me. Yet, one of the other three had offered to provide a full scholarship.",
    "subject": "I",
    "emotion_choices": [
        "Annoyance & Disappointment & Pride",
        "Annoyance & Pride & Relief",
        "Disappointment & Hopeless & Embarrassment",
        "Disappointment & Pride & Embarrassment",
        "Disappointment & Pride & Hopeless",
        "Disappointment & Relief & Hopeless"
    ],
    "emotion_label": "Annoyance & Disappointment & Pride",
    "cause_choices": [
        "I applied to a significant number of universities & I did not want to get an offer from other universities & I get to attend a university on a full scholarship",
        "I got rejected by some universities but offered a full scholarship by another & I was rejected by my dream university & I get to attend a university on a full scholarship",
        "I was rejected by my dream university & I got an offer from a low ranking university & I did not want to get an offer from other universities",
        "I applied to a significant number of universities & I got an offer from a low ranking university & I received an email instead of a call",
        "I applied to a significant number of universities & I get to attend a university on a full scholarship & I got into my dream university",
        "I applied to a significant number of universities & I got an offer from a low ranking university & I am better than the admission team"
    ],
    "cause_label": "I got rejected by some universities but offered a full scholarship by another & I was rejected by my dream university & I get to attend a university on a full scholarship"
}}

**CRITICAL REQUIREMENTS:**
1. Output ONLY the JSON array, no markdown, no code blocks, no explanations
2. Generate EXACTLY {num_items} items, each focusing on a DIFFERENT emotional moment/scenario
3. Each scenario must NOT contain explicit emotion words like 'sad', 'angry', 'happy'
4. Generate exactly 5-6 emotion choices and 5-6 cause choices per item
        "5. Emotions can be SINGLE or COMBINATIONS (use & to combine): \"Delight\" OR \"Annoyance & Disappointment & Pride\"\n",
        "6. Causes can be SINGLE or COMBINATIONS (use & to combine): \"His wife appreciated his effort\" OR \"Cause1 & Cause2 & Cause3\"\n",
        "7. Choose single vs combination based on what best fits the dialogue - be flexible!\n",
        "8. Only ONE option should be clearly best for each item\n",
        "9. Each item should be unique - focus on different parts of the dialogue\n",
        "9. Each item should be unique - focus on different parts of the dialogue\n",
**Emotion Taxonomy:**
Basic: Happiness, Sadness, Anger, Fear, Surprise, Disgust
Complex: Pride, Relief, Hope, Disappointment, Annoyance, Embarrassment, Hopeless, Delight
Social: Gratitude, Guilt, Shame, Envy, Jealousy

Output ONLY the JSON array starting with [ and ending with ]."""

    for attempt in range(max_retries):
        try:
            response = call_llm(
                prompt,
                temperature=0.4,  # Slightly higher for diversity
                max_tokens=4000  # More tokens for multiple items
            )

            if not response or len(response.strip()) == 0:
                if attempt < max_retries - 1:
                    continue
                return []

            # Extract JSON - try multiple methods
            response = response.strip()

            # Method 1: Remove markdown code blocks
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()

            # Method 2: Find JSON array boundaries
            start_idx = response.find('[')
            end_idx = response.rfind(']')
            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                response = response[start_idx:end_idx+1]

            response = response.strip()

            if not response or len(response) < 10:
                if attempt < max_retries - 1:
                    continue
                return []

            # Parse JSON array
            eu_items = json.loads(response)

            # Validate it's a list
            if not isinstance(eu_items, list):
                eu_items = [eu_items]  # If single object, wrap in list

            # Validate each item
            valid_items = []
            required_fields = [
                "scenario", "emotion_choices", "emotion_label",
                "cause_choices", "cause_label"
            ]

            for item in eu_items:
                if all(field in item for field in required_fields):
                    if len(item.get("emotion_choices", [])) >= 3 and len(item.get("cause_choices", [])) >= 3:
                        valid_items.append(item)

            if len(valid_items) > 0:
                return valid_items
            else:
                if attempt < max_retries - 1:
                    continue
                return []

        except json.JSONDecodeError:
            if attempt < max_retries - 1:
                continue
            return []
        except Exception as e:
            if attempt < max_retries - 1:
                continue
            return []

    return []


def extract_multiple_ea_items(
    dialogue: List[Tuple[str, str]],
    metadata: Dict,
    num_items: int = 3,
    max_retries: int = 3
) -> List[Dict]:
    """
    Extract MULTIPLE Emotional Application MCQs from a single dialogue.
    Returns a list of EA items (can be empty if extraction fails).
    """
    dialogue_str = "\n".join([
        f"{role.upper()}: {msg}" for role, msg in dialogue
    ])

    # Truncate dialogue if too long
    if len(dialogue_str) > 3000:
        dialogue_str = dialogue_str[:3000] + "..."

    prompt = f"""You are an expert at extracting emotional application items from therapy dialogues.

DIALOGUE:
{dialogue_str}

**Task**: Extract {num_items} DIFFERENT Emotional Application MCQs from this dialogue. Each MCQ should focus on a DIFFERENT scenario, situation, or decision point mentioned in the dialogue. You MUST output a JSON array with {num_items} items, nothing else.

**IMPORTANT**:
- Each item should focus on a DIFFERENT part of the dialogue or a DIFFERENT scenario
- Each item should be unique and distinct from the others
- The subject MUST be mentioned in each scenario

Required JSON format (array of objects):
[
    {{
        "qid": "1",
        "language": "en",
        "category": "Personal-Others" or "Social-Other" or other appropriate category,
        "question type": "Action" or "Response",
        "scenario": "A clear scenario describing someone's emotional situation. The subject MUST be mentioned in the scenario.",
        "subject": "I" or character name (MUST appear in the scenario),
        "choices": [
            "Response option 1",
            "Response option 2",
            "Response option 3",
            "Response option 4"
        ],
        "label": "The correct/empathetic response"
    }},
    {{
        "qid": "2",
        ...
    }},
    ...
]


**Examples:**

Example 1:
{{
    "qid": "12",
    "language": "en",
    "category": "Personal-Others",
    "question type": "Action",
    "scenario": "My friend lied to me about finishing the part of our group project that he was responsible for.",
    "subject": "I",
    "choices": [
        "Show your disappointment to him",
        "Carry on doing the project by yourself",
        "Acknowledge his mistake and discuss future plans",
        "Find a new teammate"
    ],
    "label": "Acknowledge his mistake and discuss future plans"
}}

Example 2:
{{
    "qid": "15",
    "language": "en",
    "category": "Personal-Others",
    "question type": "Action",
    "scenario": "Benjiro's parents are in their late 80s and living interstate in a house by themselves. He is worried that they need some help but they angrily deny it any time he brings up the subject.",
    "subject": "Benjiro",
    "choices": [
        "Frequently visit his family",
        "Believe his parents' claim that they are fine",
        "Keep telling his parents his concern, stressing their importance",
        "Move into his parents' house."
    ],
    "label": "Frequently visit his family"
}}

**CRITICAL REQUIREMENTS:**
1. Output ONLY the JSON array, no markdown, no code blocks, no explanations
2. Generate EXACTLY {num_items} items, each focusing on a DIFFERENT scenario/situation
3. Generate exactly 4 response choices per item
4. Only ONE option should be clearly best (empathetic, appropriate) for each item
5. Category must be one of: Social-Self, Social-Other, Self-Self, Self-Other, Personal-Others, or other appropriate category
        "6. **The subject MUST be mentioned in each scenario** (e.g., if subject is \"I\", scenario should say \"I\" or \"my\"; if subject is \"Benjiro\", scenario should mention \"Benjiro\")\n",
7. Question type can be "Action" (what to do) or "Response" (what to say)
8. Each item should be unique - focus on different parts of the dialogue

**Category Guide:**
- Social-Self: How to respond to your own social situation
- Social-Other: How to respond to someone else's social situation
- Self-Self: How to respond to your own personal situation
- Self-Other: How to respond to someone else's personal situation
- Personal-Others: Personal situations involving others

Output ONLY the JSON array starting with [ and ending with ]."""

    for attempt in range(max_retries):
        try:
            response = call_llm(
                prompt,
                temperature=0.4,  # Slightly higher for diversity
                max_tokens=3500  # More tokens for multiple items
            )

            if not response or len(response.strip()) == 0:
                if attempt < max_retries - 1:
                    continue
                return []

            # Extract JSON - try multiple methods
            response = response.strip()

            # Method 1: Remove markdown code blocks
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()

            # Method 2: Find JSON array boundaries
            start_idx = response.find('[')
            end_idx = response.rfind(']')
            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                response = response[start_idx:end_idx+1]

            response = response.strip()

            if not response or len(response) < 10:
                if attempt < max_retries - 1:
                    continue
                return []

            # Parse JSON array
            ea_items = json.loads(response)

            # Validate it's a list
            if not isinstance(ea_items, list):
                ea_items = [ea_items]  # If single object, wrap in list

            # Validate each item
            valid_items = []
            required_fields = [
                "scenario", "subject", "choices", "label"
            ]

            for item in ea_items:
                if all(field in item for field in required_fields):
                    # Validate subject is in scenario
                    scenario = item.get("scenario", "").lower()
                    subject = item.get("subject", "").lower()
                    if subject and (subject in scenario or "i" in subject and ("i " in scenario or " my " in scenario or " me " in scenario)):
                        if len(item.get("choices", [])) == 4:
                            valid_items.append(item)

            if len(valid_items) > 0:
                return valid_items
            else:
                if attempt < max_retries - 1:
                    continue
                return []

        except json.JSONDecodeError:
            if attempt < max_retries - 1:
                continue
            return []
        except Exception as e:
            if attempt < max_retries - 1:
                continue
            return []

    return []


print("✓ Multiple extraction functions loaded (extract_multiple_eu_items, extract_multiple_ea_items)")


✓ Multiple extraction functions loaded (extract_multiple_eu_items, extract_multiple_ea_items)


In [19]:
# Category Configuration for Targeted Generation
# Based on evaluation results - prioritize underperforming categories

# EU Categories with sampling weights (higher = more samples)
EU_CATEGORY_CONFIG = {
    "coarse_categories": {
        "personal_beliefs_and_experiences": {
            "weight": 3.0,  # PRIORITY: worst performance (0.268)
            "finegrained": ["cultural_value", "persona", "sentimental_value"]
        },
        "perspective_taking": {
            "weight": 3.0,  # PRIORITY: second worst (0.269)
            "finegrained": ["false_belief", "faux_pas", "strange_story"]
        },
        "emotional_cues": {
            "weight": 1.5,  # Medium priority (0.429)
            "finegrained": ["visual_cues", "vocal_cues"]
        },
        "complex_emotions": {
            "weight": 1.5,  # Medium priority (0.429)
            "finegrained": ["emotion_transition", "mixture_of_emotions", "unexpected_outcome"]
        }
    }
}

# EA Categories with sampling weights
EA_CATEGORY_CONFIG = {
    "categories": {
        "Social-Others": {"weight": 3.0, "types": ["Action", "Response"]},  # PRIORITY: worst degradation
        "Personal-Others": {"weight": 2.5, "types": ["Action", "Response"]},
        "Social-Self": {"weight": 2.0, "types": ["Action", "Response"]},
        "Personal-Self": {"weight": 1.5, "types": ["Action", "Response"]}
    }
}

print("✓ Category configuration loaded")
print(f"  EU categories: {list(EU_CATEGORY_CONFIG['coarse_categories'].keys())}")
print(f"  EA categories: {list(EA_CATEGORY_CONFIG['categories'].keys())}")


✓ Category configuration loaded
  EU categories: ['personal_beliefs_and_experiences', 'perspective_taking', 'emotional_cues', 'complex_emotions']
  EA categories: ['Social-Others', 'Personal-Others', 'Social-Self', 'Personal-Self']


In [20]:
# Category-Targeted Extraction Functions with Quality Filtering

import random

def sample_target_category(category_config: Dict) -> Tuple[str, str]:
    """
    Sample a category based on weights (higher weight = more likely to be sampled).
    For EU: returns (coarse_category, finegrained_category)
    For EA: returns (category, question_type)
    """
    # Weighted random sampling
    if 'coarse_categories' in category_config:  # EU
        categories = list(category_config['coarse_categories'].items())
        weights = [cat[1]['weight'] for cat in categories]
        coarse_cat = random.choices(categories, weights=weights, k=1)[0]
        coarse_name = coarse_cat[0]
        fine_cats = coarse_cat[1]['finegrained']
        fine_name = random.choice(fine_cats)
        return coarse_name, fine_name
    else:  # EA
        categories = list(category_config['categories'].items())
        weights = [cat[1]['weight'] for cat in categories]
        category = random.choices(categories, weights=weights, k=1)[0]
        cat_name = category[0]
        q_type = random.choice(category[1]['types'])
        return cat_name, q_type


def extract_targeted_eu_items(
    dialogue: List[Tuple[str, str]],
    metadata: Dict,
    num_items: int = 3,
    max_retries: int = 3
) -> List[Dict]:
    """
    Extract EU items with TARGETED categories (upsampling underperforming ones).
    """
    # Sample target categories for this dialogue
    target_categories = []
    for _ in range(num_items):
        coarse, fine = sample_target_category(EU_CATEGORY_CONFIG)
        target_categories.append((coarse, fine))

    dialogue_str = "\n".join([f"{role.upper()}: {msg}" for role, msg in dialogue])
    if len(dialogue_str) > 3000:
        dialogue_str = dialogue_str[:3000] + "..."

    # Build targeted prompt
    category_instructions = "\n".join([
        f"Item {i+1}: coarse_category='{coarse}', finegrained_category='{fine}'"
        for i, (coarse, fine) in enumerate(target_categories)
    ])

    prompt = f"""You are an expert at creating Emotional Understanding MCQs from therapy dialogues.

DIALOGUE:
{dialogue_str}

**Task**: Extract {num_items} DIFFERENT Emotional Understanding MCQs. Each MCQ should:
1. Focus on a DIFFERENT emotional moment in the dialogue
2. Match the SPECIFIC category assigned below:

{category_instructions}

**Category Definitions:**
- personal_beliefs_and_experiences: Questions about cultural values, personal identity, or sentimental attachments
- perspective_taking: Questions about understanding others' beliefs, detecting social mistakes (faux pas), or interpreting ambiguous social situations
- emotional_cues: Questions about recognizing emotions from visual cues (facial expressions, body language) or vocal cues (tone, pitch)
- complex_emotions: Questions about emotion transitions, mixed emotions, or unexpected emotional reactions

**Fine-grained Categories:**
- cultural_value: scenarios involving cultural beliefs/practices
- persona: scenarios about personal identity/character traits
- sentimental_value: scenarios about emotional attachments to objects/places
- false_belief: scenarios where someone believes something incorrect
- faux_pas: scenarios involving social mistakes/inappropriate behavior
- strange_story: ambiguous social situations requiring interpretation
- visual_cues: recognizing emotions from appearance/body language
- vocal_cues: recognizing emotions from voice tone/pitch
- emotion_transition: emotions changing in response to events
- mixture_of_emotions: multiple simultaneous emotions
- unexpected_outcome: surprising emotional reactions

Required JSON format (array of {num_items} objects):
[
    {{
        "qid": "1",
        "language": "en",
        "coarse_category": "<from list above>",
        "finegrained_category": "<from list above>",
        "scenario": "Clear scenario WITHOUT emotion words",
        "subject": "I" or character name,
        "emotion_choices": ["Emotion1", "Emotion2", ..., "Emotion5 or 6"],
        "emotion_label": "Correct emotion",
        "cause_choices": ["Cause1", "Cause2", ..., "Cause5 or 6"],
        "cause_label": "Correct cause"
    }}
]

**CRITICAL:**
1. Output ONLY JSON array
2. Match the assigned categories for each item
3. NO emotion words in scenarios
4. Emotions and causes can be single or combinations (use &)
5. 5-6 choices each
6. Only ONE clearly correct answer

Output JSON array only:"""

    for attempt in range(max_retries):
        try:
            response = call_llm(prompt, temperature=0.4, max_tokens=4000)
            if not response or len(response.strip()) == 0:
                if attempt < max_retries - 1:
                    continue
                return []

            response = response.strip()
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()

            start_idx = response.find('[')
            end_idx = response.rfind(']')
            if start_idx != -1 and end_idx != -1:
                response = response[start_idx:end_idx+1]

            eu_items = json.loads(response)
            if not isinstance(eu_items, list):
                eu_items = [eu_items]

            # Quality filtering
            valid_items = []
            for item in eu_items:
                # Check required fields
                required = ["scenario", "emotion_choices", "emotion_label", "cause_choices", "cause_label", "coarse_category", "finegrained_category"]
                if not all(field in item for field in required):
                    continue

                # Quality checks
                scenario = item.get("scenario", "")
                emotion_words = ['sad', 'happy', 'angry', 'fear', 'disgust', 'surprise', 'joy', 'delight', 'disappointed']
                if any(word in scenario.lower() for word in emotion_words):
                    continue  # Skip if has explicit emotion words

                if len(item.get("emotion_choices", [])) < 4 or len(item.get("cause_choices", [])) < 4:
                    continue  # Skip if too few choices

                if len(scenario) < 50:
                    continue  # Skip if scenario too short

                valid_items.append(item)

            if len(valid_items) > 0:
                return valid_items

            if attempt < max_retries - 1:
                continue
            return []

        except Exception as e:
            if attempt < max_retries - 1:
                continue
            return []

    return []


def extract_targeted_ea_items(
    dialogue: List[Tuple[str, str]],
    metadata: Dict,
    num_items: int = 3,
    max_retries: int = 3
) -> List[Dict]:
    """
    Extract EA items with TARGETED categories (upsampling underperforming ones).
    """
    # Sample target categories
    target_categories = []
    for _ in range(num_items):
        category, q_type = sample_target_category(EA_CATEGORY_CONFIG)
        target_categories.append((category, q_type))

    dialogue_str = "\n".join([f"{role.upper()}: {msg}" for role, msg in dialogue])
    if len(dialogue_str) > 3000:
        dialogue_str = dialogue_str[:3000] + "..."

    category_instructions = "\n".join([
        f"Item {i+1}: category='{cat}', question_type='{qtype}'"
        for i, (cat, qtype) in enumerate(target_categories)
    ])

    prompt = f"""You are an expert at creating Emotional Application MCQs from therapy dialogues.

DIALOGUE:
{dialogue_str}

**Task**: Extract {num_items} DIFFERENT Emotional Application MCQs. Each MCQ should:
1. Focus on a DIFFERENT scenario from the dialogue
2. Match the SPECIFIC category assigned below:

{category_instructions}

**Category Definitions:**
- Personal-Others: How to handle personal situations involving other people (family, friends, coworkers)
- Social-Others: How to respond to someone else's social dilemma or social situation
- Personal-Self: How to handle your own personal emotional situation
- Social-Self: How to handle your own social situation

**Question Types:**
- Action: What should the person DO?
- Response: What should the person SAY?

Required JSON format (array of {num_items} objects):
[
    {{
        "qid": "1",
        "language": "en",
        "category": "<from list above>",
        "question type": "Action or Response",
        "scenario": "Clear scenario - subject MUST be mentioned",
        "subject": "I" or character name (MUST appear in scenario),
        "choices": ["Option 1", "Option 2", "Option 3", "Option 4"],
        "label": "The emotionally intelligent response"
    }}
]

**CRITICAL:**
1. Output ONLY JSON array
2. Match assigned categories
3. Subject MUST be mentioned in scenario
4. Exactly 4 choices
5. Only ONE clearly best answer (emotionally intelligent, empathetic)
6. Make scenarios realistic and nuanced

Output JSON array only:"""

    for attempt in range(max_retries):
        try:
            response = call_llm(prompt, temperature=0.4, max_tokens=3500)
            if not response or len(response.strip()) == 0:
                if attempt < max_retries - 1:
                    continue
                return []

            response = response.strip()
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()

            start_idx = response.find('[')
            end_idx = response.rfind(']')
            if start_idx != -1 and end_idx != -1:
                response = response[start_idx:end_idx+1]

            ea_items = json.loads(response)
            if not isinstance(ea_items, list):
                ea_items = [ea_items]

            # Quality filtering
            valid_items = []
            for item in ea_items:
                required = ["scenario", "subject", "choices", "label", "category"]
                if not all(field in item for field in required):
                    continue

                # Validate subject is mentioned
                scenario = item.get("scenario", "").lower()
                subject = item.get("subject", "").lower()
                if subject and not (subject in scenario or ("i" in subject and ("i " in scenario or " my " in scenario or " me " in scenario))):
                    continue

                if len(item.get("choices", [])) != 4:
                    continue

                if len(scenario) < 30:
                    continue

                valid_items.append(item)

            if len(valid_items) > 0:
                return valid_items

            if attempt < max_retries - 1:
                continue
            return []

        except Exception as e:
            if attempt < max_retries - 1:
                continue
            return []

    return []


print("✓ Category-targeted extraction functions loaded")
print("  - extract_targeted_eu_items: generates EU items with category upsampling")
print("  - extract_targeted_ea_items: generates EA items with category upsampling")
print("  - Quality filtering enabled (rejects poor generations)")


✓ Category-targeted extraction functions loaded
  - extract_targeted_eu_items: generates EU items with category upsampling
  - extract_targeted_ea_items: generates EA items with category upsampling
  - Quality filtering enabled (rejects poor generations)


## Agent Classes



In [21]:
class BackgroundGeneratorAgent:
    """
    Generates elaborate, non-repetitive backgrounds for patients.
    Avoids stale themes like generic work stress/exams/breakups.
    """

    def __init__(self, themes: List[Dict]):
        self.themes = themes
        self.used_combinations = set()

    def generate_background(
        self,
        persona: str,
        theme: Optional[Dict] = None,
        avoid_generic: bool = True
    ) -> str:
        """
        Generate an elaborate background that combines:
        - The persona's characteristics
        - A specific theme/category
        - Unique circumstances, relationships, history
        - Cultural/social context
        - Recent events that set up emotional complexity
        """
        if theme is None:
            theme = random.choice(self.themes)

        # Build a unique combination key to avoid repetition
        combo_key = f"{persona[:50]}_{theme['theme']}"

        prompt = f"""You are a background generator for creating rich, psychologically complex patient backgrounds for therapy simulations.

Persona: {persona}
Theme Category: {theme['category']}
Theme: {theme['theme']}
Description: {theme['description']}

Generate an elaborate, detailed background (300-500 words) that includes:

1. **Personal History**: Specific events, relationships, and experiences that shaped this person
2. **Current Life Context**: Job, living situation, family structure, social circle
3. **Cultural/Social Background**: Cultural identity, community ties, social class, geographic context
4. **Recent Triggering Event**: A specific recent event (within last few weeks) that has created emotional complexity
5. **Internal Conflicts**: Conflicting beliefs, values, or desires that create tension
6. **Relationships**: Key relationships (family, friends, romantic, professional) with specific dynamics
7. **Emotional Patterns**: How this person typically processes emotions, their emotional vocabulary

**CRITICAL CONSTRAINTS:**
- Avoid generic themes: NO generic work stress, exam anxiety, or standard breakups
- Make it SPECIFIC: Use concrete details, names, places, dates, specific circumstances
- Create COMPLEXITY: Multiple overlapping concerns, not just one simple problem
- Include CULTURAL CONTEXT: How their background/culture influences their emotional experience
- Set up for MIXED EMOTIONS: The situation should naturally lead to conflicting feelings
- Make it UNIQUE: Avoid clichés and create a distinctive narrative

Output ONLY the background text, no meta-commentary."""

        background = call_llm(
            prompt,
            temperature=0.9,  # Higher temperature for creativity
            max_tokens=1500
        )

        self.used_combinations.add(combo_key)
        return background

print("✓ BackgroundGeneratorAgent defined")



✓ BackgroundGeneratorAgent defined


In [22]:
# Improved extraction functions with retry logic and better error handling
# This replaces the extraction functions in the cell below

def extract_eu_item_improved(dialogue: List[Tuple[str, str]], metadata: Dict, max_retries: int = 3) -> Optional[Dict]:
    """
    Extract ONE Emotional Understanding MCQ from dialogue.
    Uses strict EmoBench JSON schema with retry logic.
    """
    dialogue_str = "\n".join([
        f"{role.upper()}: {msg}" for role, msg in dialogue
    ])

    # Truncate dialogue if too long
    if len(dialogue_str) > 2000:
        dialogue_str = dialogue_str[:2000] + "..."

    prompt = f"""You are an expert at extracting emotional understanding items from therapy dialogues.

DIALOGUE:
{dialogue_str}

**Task**: Create ONE Emotional Understanding MCQ following the EXACT EmoBench JSON schema. You MUST output ONLY valid JSON, nothing else.

**IMPORTANT**: Emotions can be SINGLE emotions OR combinations. Choose what best fits the dialogue:
- Single emotions: "Delight", "Anger", "Pride", "Disappointment"
- Combinations: "Annoyance & Disappointment & Pride", "Happiness & Relief"

Required JSON format:
{{
    "qid": "1",
    "language": "en",
    "coarse_category": "complex_emotions",
    "finegrained_category": "mixture_of_emotions" or "emotion_transition" or other appropriate category,
    "scenario": "A clear scenario WITHOUT explicit emotion words (no 'sad', 'angry', etc.)",
    "subject": "I" or character name,
    "emotion_choices": [
        "Single emotion OR Emotion1 & Emotion2",
        "Single emotion OR Emotion2 & Emotion3",
        "Single emotion OR Emotion3 & Emotion4",
        "Single emotion OR Emotion4 & Emotion5",
        "Single emotion OR Emotion5 & Emotion6"
    ],
    "emotion_label": "The correct emotion (single or combination)",
    "cause_choices": [
        "Single cause OR Cause1 & Cause2",
        "Single cause OR Cause2 & Cause3",
        "Single cause OR Cause3 & Cause4",
        "Single cause OR Cause4 & Cause5",
        "Single cause OR Cause5 & Cause6"
    ],
    "cause_label": "The correct cause (single or combination)"
}}

**Examples:**

Example 1 (Single Emotion):
{{
    "qid": "2",
    "language": "en",
    "coarse_category": "complex_emotions",
    "finegrained_category": "emotion_transition",
    "scenario": "Simon, an amateur painter, had been working on a portrait of his deceased pet for over a week. After he finally completed it, he stepped back and immediately noticed that he had painted the wrong color for his pet's fur. Just then, his wife saw it and exclaimed, \"That's exactly how I remember him!\"",
    "subject": "Simon",
    "emotion_choices": [
        "Delight",
        "Anger",
        "Embarrassment",
        "Hopeless",
        "Pride",
        "Disappointment"
    ],
    "emotion_label": "Delight",
    "cause_choices": [
        "He thought he had painted the perfect portrait",
        "His wife appreciated his effort and liked his portrait",
        "He made a mistake in a painting he cared about",
        "His wife appreciated his effort despite noticing the mistake"
    ],
    "cause_label": "His wife appreciated his effort and liked his portrait"
}}

Example 2 (Combination):
{{
    "qid": "21",
    "language": "en",
    "coarse_category": "complex_emotions",
    "finegrained_category": "mixture_of_emotions",
    "scenario": "I applied to eight different universities to pursue my Bachelor's degree. Each decision letter came in differing dates. I waited until I received all the decision letters and opened them simultaneously. After opening the letters, I found that five of the universities, including my dream university, rejected me. Yet, one of the other three had offered to provide a full scholarship.",
    "subject": "I",
    "emotion_choices": [
        "Annoyance & Disappointment & Pride",
        "Annoyance & Pride & Relief",
        "Disappointment & Hopeless & Embarrassment",
        "Disappointment & Pride & Embarrassment",
        "Disappointment & Pride & Hopeless",
        "Disappointment & Relief & Hopeless"
    ],
    "emotion_label": "Annoyance & Disappointment & Pride",
    "cause_choices": [
        "I applied to a significant number of universities & I did not want to get an offer from other universities & I get to attend a university on a full scholarship",
        "I got rejected by some universities but offered a full scholarship by another & I was rejected by my dream university & I get to attend a university on a full scholarship",
        "I was rejected by my dream university & I got an offer from a low ranking university & I did not want to get an offer from other universities",
        "I applied to a significant number of universities & I got an offer from a low ranking university & I received an email instead of a call",
        "I applied to a significant number of universities & I get to attend a university on a full scholarship & I got into my dream university",
        "I applied to a significant number of universities & I got an offer from a low ranking university & I am better than the admission team"
    ],
    "cause_label": "I got rejected by some universities but offered a full scholarship by another & I was rejected by my dream university & I get to attend a university on a full scholarship"
}}

**CRITICAL REQUIREMENTS:**
1. Output ONLY the JSON object, no markdown, no code blocks, no explanations
2. Scenario must NOT contain explicit emotion words like 'sad', 'angry', 'happy'
3. Generate exactly 5-6 emotion choices and 5-6 cause choices
4. Emotions can be SINGLE or COMBINATIONS (use & to combine): "Delight" OR "Annoyance & Disappointment & Pride"
5. Causes can be SINGLE or COMBINATIONS (use & to combine): "His wife appreciated his effort" OR "Cause1 & Cause2 & Cause3"
6. Choose single vs combination based on what best fits the dialogue - be flexible!
7. Only ONE option should be clearly best

**Emotion Taxonomy:**
Basic: Happiness, Sadness, Anger, Fear, Surprise, Disgust
Complex: Pride, Relief, Hope, Disappointment, Annoyance, Embarrassment, Hopeless, Delight
Social: Gratitude, Guilt, Shame, Envy, Jealousy

Output ONLY the JSON object starting with {{ and ending with }}."""

    for attempt in range(max_retries):
        try:
            response = call_llm(
                prompt,
                temperature=0.3,  # Lower temperature for more consistent JSON
                max_tokens=2000
            )

            if not response or len(response.strip()) == 0:
                if attempt < max_retries - 1:
                    continue
                return None

            # Extract JSON - try multiple methods
            response = response.strip()

            # Method 1: Remove markdown code blocks
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()

            # Method 2: Find JSON object boundaries
            start_idx = response.find('{')
            end_idx = response.rfind('}')
            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                response = response[start_idx:end_idx+1]

            response = response.strip()

            if not response or len(response) < 10:
                if attempt < max_retries - 1:
                    continue
                return None

            # Parse JSON
            eu_item = json.loads(response)

            # Validate required fields
            required_fields = [
                "scenario", "emotion_choices", "emotion_label",
                "cause_choices", "cause_label"
            ]
            if all(field in eu_item for field in required_fields):
                # Additional validation
                if len(eu_item.get("emotion_choices", [])) >= 3 and len(eu_item.get("cause_choices", [])) >= 3:
                    return eu_item
                else:
                    if attempt < max_retries - 1:
                        continue
                    return None
            else:
                if attempt < max_retries - 1:
                    continue
                return None

        except json.JSONDecodeError:
            if attempt < max_retries - 1:
                continue
            return None
        except Exception:
            if attempt < max_retries - 1:
                continue
            return None

    return None


def extract_ea_item_improved(dialogue: List[Tuple[str, str]], metadata: Dict, max_retries: int = 3) -> Optional[Dict]:
    """
    Extract ONE Emotional Application MCQ from dialogue.
    Uses strict EmoBench JSON schema with retry logic.
    """
    dialogue_str = "\n".join([
        f"{role.upper()}: {msg}" for role, msg in dialogue
    ])

    # Truncate dialogue if too long
    if len(dialogue_str) > 2000:
        dialogue_str = dialogue_str[:2000] + "..."

    prompt = f"""You are an expert at extracting emotional application items from therapy dialogues.

DIALOGUE:
{dialogue_str}

**Task**: Create ONE Emotional Application MCQ following the EXACT EmoBench JSON schema. You MUST output ONLY valid JSON, nothing else.

Required JSON format:
{{
    "qid": "1",
    "language": "en",
    "category": "Personal-Others" or "Social-Other" or other appropriate category,
    "question type": "Action" or "Response",
    "scenario": "A clear scenario describing someone's emotional situation. The subject MUST be mentioned in the scenario.",
    "subject": "I" or character name (MUST appear in the scenario),
    "choices": [
        "\\"Response option 1\\"",
        "\\"Response option 2\\"",
        "\\"Response option 3\\"",
        "\\"Response option 4\\""
    ],
    "label": "The correct/empathetic response"
}}

**Examples:**

Example 1:
{{
    "qid": "12",
    "language": "en",
    "category": "Personal-Others",
    "question type": "Action",
    "scenario": "My friend lied to me about finishing the part of our group project that he was responsible for.",
    "subject": "I",
    "choices": [
        "Show your disappointment to him",
        "Carry on doing the project by yourself",
        "Acknowledge his mistake and discuss future plans",
        "Find a new teammate"
    ],
    "label": "Acknowledge his mistake and discuss future plans"
}}

Example 2:
{{
    "qid": "15",
    "language": "en",
    "category": "Personal-Others",
    "question type": "Action",
    "scenario": "Benjiro's parents are in their late 80s and living interstate in a house by themselves. He is worried that they need some help but they angrily deny it any time he brings up the subject.",
    "subject": "Benjiro",
    "choices": [
        "Frequently visit his family",
        "Believe his parents' claim that they are fine",
        "Keep telling his parents his concern, stressing their importance",
        "Move into his parents' house."
    ],
    "label": "Frequently visit his family"
}}

**CRITICAL REQUIREMENTS:**
1. Output ONLY the JSON object, no markdown, no code blocks, no explanations
2. Generate exactly 4 response choices
3. Only ONE option should be clearly best (empathetic, appropriate)
4. Category must be one of: Social-Self, Social-Other, Self-Self, Self-Other, Personal-Others, or other appropriate category
5. **The subject MUST be mentioned in the scenario** (e.g., if subject is "I", scenario should say "I" or "my"; if subject is "Benjiro", scenario should mention "Benjiro")
6. Question type can be "Action" (what to do) or "Response" (what to say)

**Category Guide:**
- Social-Self: How to respond to your own social situation
- Social-Other: How to respond to someone else's social situation
- Self-Self: How to respond to your own personal situation
- Self-Other: How to respond to someone else's personal situation
- Personal-Others: Personal situations involving others

Output ONLY the JSON object starting with {{ and ending with }}."""

    for attempt in range(max_retries):
        try:
            response = call_llm(
                prompt,
                temperature=0.3,  # Lower temperature for more consistent JSON
                max_tokens=1500
            )

            if not response or len(response.strip()) == 0:
                if attempt < max_retries - 1:
                    continue
                return None

            # Extract JSON - try multiple methods
            response = response.strip()

            # Method 1: Remove markdown code blocks
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()

            # Method 2: Find JSON object boundaries
            start_idx = response.find('{')
            end_idx = response.rfind('}')
            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                response = response[start_idx:end_idx+1]

            response = response.strip()

            if not response or len(response) < 10:
                if attempt < max_retries - 1:
                    continue
                return None

            # Parse JSON
            ea_item = json.loads(response)

            # Validate required fields
            required_fields = ["scenario", "choices", "label", "subject"]
            if all(field in ea_item for field in required_fields):
                # Additional validation
                if len(ea_item.get("choices", [])) >= 3:
                    # Check that subject appears in scenario
                    subject = ea_item.get("subject", "").strip()
                    scenario = ea_item.get("scenario", "").strip()

                    if subject and scenario:
                        # Check if subject appears in scenario
                        # Handle "I" case - check for "I", "my", "me", "myself"
                        if subject.lower() == "i":
                            subject_in_scenario = any(word in scenario.lower() for word in [" i ", " i,", " i.", " i'", "my ", "me ", "myself"])
                        else:
                            # For named subjects, check if the name appears in scenario
                            subject_in_scenario = subject in scenario or subject.lower() in scenario.lower()

                        if not subject_in_scenario:
                            if attempt < max_retries - 1:
                                continue
                            return None

                    return ea_item
                else:
                    if attempt < max_retries - 1:
                        continue
                    return None
            else:
                if attempt < max_retries - 1:
                    continue
                return None

        except json.JSONDecodeError:
            if attempt < max_retries - 1:
                continue
            return None
        except Exception:
            if attempt < max_retries - 1:
                continue
            return None

    return None

# Override the original functions
extract_eu_item = extract_eu_item_improved
extract_ea_item = extract_ea_item_improved

print("✓ Improved extraction functions loaded (with retry logic)")


✓ Improved extraction functions loaded (with retry logic)


In [23]:
class ClientAgent:
    """
    Client/Patient agent that absorbs persona + background and engages authentically.
    Task: Describe concrete recent event + feelings + conflicting thoughts.
    """

    def __init__(self, persona: str, background: str):
        self.persona = persona
        self.background = background
        self.dialogue_history = []
        self.emotional_state = {}

    def initial_statement(self) -> str:
        """
        Generate the client's opening statement.
        Must include: concrete recent event + feelings + at least one conflicting thought.
        """
        prompt = f"""You are a therapy client. You have the following persona and background:

PERSONA: {self.persona}

BACKGROUND:
{self.background}

**Your Task**: Create your opening statement to a therapist. This should be natural, authentic, and include:

1. A CONCRETE RECENT EVENT (specific, happened recently, with details)
2. How you FEEL about it (but don't use explicit emotion words like "sad" or "angry" - describe the feeling)
3. At least ONE CONFLICTING THOUGHT or mixed feeling (e.g., "part of me feels X but another part feels Y")
4. Some context about why this matters to you personally

**Style**: Write as if you're speaking to a therapist for the first time. Be genuine, maybe a bit hesitant or uncertain. Don't over-explain or be too analytical - just share what's on your mind.

Output ONLY your opening statement (2-4 sentences), no meta-commentary."""

        statement = call_llm(
            prompt,
            temperature=0.8,
            max_tokens=300
        )

        self.dialogue_history.append(("client", statement))
        return statement

    def respond(self, therapist_message: str) -> str:
        """
        Respond to therapist's message based on persona + background + dialogue history.
        """
        history_str = "\n".join([
            f"{role.upper()}: {msg}" for role, msg in self.dialogue_history[-6:]
        ])

        prompt = f"""You are a therapy client. You have the following persona and background:

PERSONA: {self.persona}

BACKGROUND:
{self.background}

RECENT DIALOGUE:
{history_str}

THERAPIST JUST SAID:
{therapist_message}

**Your Task**: Respond authentically as this person would. Consider:
- Your persona and background inform your perspective
- You're in therapy, so you're trying to be honest but maybe guarded
- The therapist's question might make you think about things differently
- You might reveal new information or feelings as you talk

**Style**: Natural, conversational, authentic to your character. 1-3 sentences typically.

Output ONLY your response, no meta-commentary."""

        response = call_llm(
            prompt,
            temperature=0.8,
            max_tokens=200
        )

        self.dialogue_history.append(("client", response))
        return response

print("✓ ClientAgent defined")



✓ ClientAgent defined


In [26]:
class TherapistAgent:
    """
    Therapist agent with structured turn-by-turn strategy:
    - Turns 2-3: Clarify facts vs feelings
    - Turns 4-5: Probe personal beliefs, sentimental value, culture
    - Turns 6-7: Explore alternative perspectives and possible responses
    Each turn must introduce new information.
    """

    def __init__(self):
        self.dialogue_history = []
        self.turn_count = 0
        self.strategy_phase = "initial"  # initial, clarification, beliefs, perspectives

    def respond(self, client_message: str, turn_number: int) -> str:
        """
        Generate therapist response based on turn number and strategy phase.
        """
        self.turn_count = turn_number
        self.dialogue_history.append(("client", client_message))

        # Determine strategy phase
        if turn_number <= 2:
            phase = "initial_clarification"
            strategy = "Clarify basic facts vs feelings. Help the client distinguish between what happened and how they feel about it."
        elif turn_number <= 4:
            phase = "beliefs_culture"
            strategy = "Probe personal beliefs, values, cultural background, and sentimental attachments. Explore what this situation means to them personally and culturally."
        else:
            phase = "perspectives_responses"
            strategy = "Explore alternative perspectives, possible responses, and what the client might do or say. Help them see different angles."

        history_str = "\n".join([
            f"{role.upper()}: {msg}" for role, msg in self.dialogue_history[-8:]
        ])

        prompt = f"""You are a skilled therapist conducting a therapy session. Your current strategy phase: {phase}

STRATEGY FOR THIS PHASE: {strategy}

DIALOGUE HISTORY:
{history_str}

**Your Task**: Generate a therapist response that:
1. Follows the strategy for this phase
2. Introduces NEW information or perspective (not just validation)
3. Is empathetic but probing
4. Helps surface implicit beliefs, mixed emotions, or dilemmas
5. Moves the conversation forward meaningfully

**Style**: Professional, warm, thoughtful. 1-2 sentences typically. Ask open-ended questions that invite reflection.

Output ONLY your response, no meta-commentary."""

        response = call_llm(
            prompt,
            temperature=0.7,
            max_tokens=200
        )

        self.dialogue_history.append(("therapist", response))
        return response

print("✓ TherapistAgent defined")



✓ TherapistAgent defined


In [27]:
# Set output directory to save in your project folder
# This ensures generated data is saved to Google Drive or uploaded files location

# if 'PROJECT_PATH' in globals():
#     OUTPUT_DIR_BASE = PROJECT_PATH
# else:
#     OUTPUT_DIR_BASE = "/content"  # Fallback to current directory
#     print("⚠ Using fallback output directory. Run file access setup cells above to save to Drive/upload folder.")

# print(f"Output will be saved to: {OUTPUT_DIR_BASE}/generated_data")
OUTPUT_DIR_BASE="/content/drive/MyDrive/685_Project_New/generated_data/mads/new_generations"


**Note:** In the Configuration cell below, if you want to save outputs to your Google Drive or uploaded files location, change:
```python
OUTPUT_DIR = "generated_data"
```
to:
```python
OUTPUT_DIR = os.path.join(OUTPUT_DIR_BASE, "generated_data")
```
Or manually set it to your desired path.


In [28]:
class SupervisorAgent:
    """
    Supervisor/Orchestrator that monitors dialogue quality.
    Checks after every 2 turns:
    - Is there a clearly identifiable scenario?
    - Are there identifiable emotions and causes?
    - Is there an emerging dilemma / "what should I do or say?" moment?
    Stops when enough structure exists (≥1 clear EU + ≥1 clear EA candidate).
    """

    def __init__(self, min_turns: int = 4, max_turns: int = 14):
        self.min_turns = min_turns
        self.max_turns = max_turns

    def evaluate_dialogue(
        self,
        dialogue: List[Tuple[str, str]],
        turn_count: int
    ) -> Tuple[bool, Dict]:
        """
        Evaluate if dialogue has enough structure for extraction.
        Returns: (should_continue, evaluation_dict)
        """
        if turn_count < self.min_turns:
            return True, {"reason": "Minimum turns not reached"}

        if turn_count >= self.max_turns:
            return False, {"reason": "Maximum turns reached"}

        # Convert dialogue to string
        dialogue_str = "\n".join([
            f"{role.upper()}: {msg}" for role, msg in dialogue
        ])

        prompt = f"""You are a supervisor evaluating a therapy dialogue for quality and extractability.

DIALOGUE:
{dialogue_str}

**Evaluate the dialogue on these criteria:**

1. **Identifiable Scenario**: Is there a clear, concrete scenario/situation described? (Yes/No)
2. **Emotions Present**: Are there identifiable emotions (even if implicit) that could be extracted? (Yes/No)
3. **Causes Present**: Are there identifiable causes or reasons for the emotions? (Yes/No)
4. **Dilemma/Application**: Is there a clear "what should I do or say?" moment or dilemma? (Yes/No)
5. **Richness**: Is the dialogue rich enough to extract both an Emotional Understanding (EU) and Emotional Application (EA) item? (Yes/No)

**Output Format (JSON):**
{{
    "should_continue": true/false,
    "scenario_clear": true/false,
    "emotions_present": true/false,
    "causes_present": true/false,
    "dilemma_present": true/false,
    "rich_enough": true/false,
    "reason": "brief explanation"
}}

Output ONLY valid JSON, no other text."""

        response = call_llm(
            prompt,
            temperature=0.3,  # Lower temperature for evaluation
            max_tokens=300
        )

        try:
            # Extract JSON from response
            response = response.strip()
            if response.startswith("```json"):
                response = response[7:]
            if response.startswith("```"):
                response = response[3:]
            if response.endswith("```"):
                response = response[:-3]
            response = response.strip()

            eval_dict = json.loads(response)
            should_continue = eval_dict.get("should_continue", True)

            # Override: if all criteria met, stop
            if all([
                eval_dict.get("scenario_clear", False),
                eval_dict.get("emotions_present", False),
                eval_dict.get("causes_present", False),
                eval_dict.get("dilemma_present", False),
                eval_dict.get("rich_enough", False)
            ]):
                should_continue = False
                eval_dict["reason"] = "All quality criteria met"

            return should_continue, eval_dict

        except json.JSONDecodeError:
            # Fallback: continue if unclear
            return turn_count < self.max_turns, {"reason": "Evaluation parse error, continuing"}

print("✓ SupervisorAgent defined")



✓ SupervisorAgent defined


## Extraction Stage (Single-Agent)



In [29]:
def extract_eu_item(dialogue: List[Tuple[str, str]], metadata: Dict) -> Optional[Dict]:
    """
    Extract ONE Emotional Understanding MCQ from dialogue.
    Uses strict EmoBench JSON schema.
    """
    dialogue_str = "\n".join([
        f"{role.upper()}: {msg}" for role, msg in dialogue
    ])

    prompt = f"""You are an expert at extracting emotional understanding items from therapy dialogues.

DIALOGUE:
{dialogue_str}

METADATA:
{json.dumps(metadata, indent=2)}

**Task**: Create ONE Emotional Understanding MCQ following the EXACT EmoBench JSON schema:

{{
    "qid": "unique_id",
    "language": "en",
    "coarse_category": "one of: basic_emotions, complex_emotions, social_emotions, self_conscious_emotions",
    "finegrained_category": "specific subcategory",
    "scenario": "A clear scenario WITHOUT explicit emotion words (no 'sad', 'angry', etc.)",
    "subject": "I",
    "emotion_choices": [
        "Emotion1 & Emotion2 & Emotion3",
        "Emotion2 & Emotion3 & Emotion4",
        ...
    ],
    "emotion_label": "The correct emotion combination",
    "cause_choices": [
        "Cause1 & Cause2 & Cause3",
        "Cause2 & Cause3 & Cause4",
        ...
    ],
    "cause_label": "The correct cause combination"
}}

**CRITICAL REQUIREMENTS:**
1. Scenario must NOT contain explicit emotion words
2. Emotion choices should include mixed/compound emotions (use & to combine)
3. Cause choices should be specific, plausible causes
4. Only ONE option should be clearly best (like EmoBench)
5. Generate 5-6 choices for both emotions and causes
6. Use realistic emotion combinations from EmoBench taxonomy

**Emotion Taxonomy (examples):**
Basic: Happiness, Sadness, Anger, Fear, Surprise, Disgust
Complex: Pride, Relief, Hope, Disappointment, Annoyance, Embarrassment, Hopeless
Social: Gratitude, Guilt, Shame, Envy, Jealousy
Self-conscious: Pride, Shame, Embarrassment

Output ONLY valid JSON, no other text."""

    response = call_llm(
        prompt,
        temperature=0.5,
        max_tokens=2000
    )

    try:
        # Extract JSON
        response = response.strip()
        if response.startswith("```json"):
            response = response[7:]
        if response.startswith("```"):
            response = response[3:]
        if response.endswith("```"):
            response = response[:-3]
        response = response.strip()

        eu_item = json.loads(response)

        # Validate required fields
        required_fields = [
            "scenario", "emotion_choices", "emotion_label",
            "cause_choices", "cause_label"
        ]
        if all(field in eu_item for field in required_fields):
            return eu_item
        else:
            return None

    except json.JSONDecodeError as e:
        print(f"EU extraction JSON error: {e}")
        return None


def extract_ea_item(dialogue: List[Tuple[str, str]], metadata: Dict) -> Optional[Dict]:
    """
    Extract ONE Emotional Application MCQ from dialogue.
    Uses strict EmoBench JSON schema.
    """
    dialogue_str = "\n".join([
        f"{role.upper()}: {msg}" for role, msg in dialogue
    ])

    prompt = f"""You are an expert at extracting emotional application items from therapy dialogues.

DIALOGUE:
{dialogue_str}

METADATA:
{json.dumps(metadata, indent=2)}

**Task**: Create ONE Emotional Application MCQ following the EXACT EmoBench JSON schema:

{{
    "qid": "unique_id",
    "language": "en",
    "category": "one of: Social-Self, Social-Other, Self-Self, Self-Other",
    "question type": "Response",
    "scenario": "A clear scenario describing someone's emotional situation",
    "subject": "I",
    "choices": [
        "\\"Response option 1\\"",
        "\\"Response option 2\\"",
        "\\"Response option 3\\"",
        "\\"Response option 4\\""
    ],
    "label": "The correct/empathetic response"
}}

**CRITICAL REQUIREMENTS:**
1. Scenario should describe someone's emotional situation clearly
2. Choices should be different response options (what to say/do)
3. Only ONE option should be clearly best (empathetic, appropriate)
4. Generate 4 choices typically
5. Category should match the type of relationship (Social-Self, Social-Other, etc.)

**Category Guide:**
- Social-Self: How to respond to your own social situation
- Social-Other: How to respond to someone else's social situation
- Self-Self: How to respond to your own personal situation
- Self-Other: How to respond to someone else's personal situation

Output ONLY valid JSON, no other text."""

    response = call_llm(
        prompt,
        temperature=0.5,
        max_tokens=1500
    )

    try:
        # Extract JSON
        response = response.strip()
        if response.startswith("```json"):
            response = response[7:]
        if response.startswith("```"):
            response = response[3:]
        if response.endswith("```"):
            response = response[:-3]
        response = response.strip()

        ea_item = json.loads(response)

        # Validate required fields
        required_fields = ["scenario", "choices", "label"]
        if all(field in ea_item for field in required_fields):
            return ea_item
        else:
            return None

    except json.JSONDecodeError as e:
        print(f"EA extraction JSON error: {e}")
        return None

print("✓ Extraction functions defined")



✓ Extraction functions defined


In [30]:
def generate_dialogue_and_extract(
    persona: Dict,
    theme: Optional[Dict] = None,
    qid_counter: int = 1,
    num_eu_items: int = 3,
    num_ea_items: int = 3
) -> Tuple[List[Dict], List[Dict], Dict, int]:
    """
    Full pipeline: Generate dialogue and extract MULTIPLE EU/EA items.
    Returns: (eu_items_list, ea_items_list, metadata, next_qid_counter)
    """
    # Step 1: Generate background
    bg_agent = BackgroundGeneratorAgent(themes)
    background = bg_agent.generate_background(
        persona=persona.get("persona", ""),
        theme=theme
    )

    # Step 2: Initialize agents
    client = ClientAgent(
        persona=persona.get("persona", ""),
        background=background
    )
    therapist = TherapistAgent()
    supervisor = SupervisorAgent()

    # Step 3: Generate dialogue
    dialogue = []
    turn_count = 0

    # Client opening
    client_opening = client.initial_statement()
    dialogue.append(("client", client_opening))
    turn_count += 1

    # Therapist-client exchange
    should_continue = True
    while should_continue and turn_count < supervisor.max_turns:
        # Get last client message for therapist to respond to
        last_client_msg = client_opening if turn_count == 1 else [msg for role, msg in dialogue if role == "client"][-1]

        # Therapist responds
        therapist_response = therapist.respond(last_client_msg, turn_count)
        dialogue.append(("therapist", therapist_response))
        turn_count += 1

        # Client responds
        client_response = client.respond(therapist_response)
        dialogue.append(("client", client_response))
        turn_count += 1

        # Supervisor evaluation (every 2 turns)
        if turn_count % 2 == 0:
            should_continue, eval_dict = supervisor.evaluate_dialogue(dialogue, turn_count)
            if not should_continue:
                break

    # Step 4: Extract MULTIPLE EU and EA items from the dialogue
    metadata = {
        "persona": persona.get("persona", ""),
        "theme": theme.get("theme", "") if theme else None,
        "background": background,
        "dialogue_turns": turn_count,
        "dialogue": dialogue
    }

    # Extract multiple EU items
    eu_items = extract_targeted_eu_items(dialogue, metadata, num_items=num_eu_items)

    # Extract multiple EA items
    ea_items = extract_targeted_ea_items(dialogue, metadata, num_items=num_ea_items)

    # Add qids to all items
    for item in eu_items:
        item["qid"] = str(qid_counter)
        qid_counter += 1

    for item in ea_items:
        item["qid"] = str(qid_counter)
        qid_counter += 1

    return eu_items, ea_items, metadata, qid_counter

print("✓ Main pipeline function defined (now extracts multiple items per dialogue)")



✓ Main pipeline function defined (now extracts multiple items per dialogue)


## Large-Scale Generation Configuration

**Key Changes for 1000-2000 samples:**
- `NUM_DIALOGUES = 2000` (generates ~2000 dialogues)
- **Multiple Items Per Dialogue**: Extracts `NUM_EU_ITEMS_PER_DIALOGUE` (default: 3) EU items and `NUM_EA_ITEMS_PER_DIALOGUE` (default: 3) EA items from each dialogue
- **Expected Output**: With 2000 dialogues and 3 items each, expect ~4000-5000 EU items and ~4000-5000 EA items (some extractions may fail)
- **Checkpointing**: Saves progress every 50 dialogues - you can stop and resume
- **Progress tracking**: Shows ETA and generation rate
- **Persona Diversity**: Tracks and ensures diverse persona sampling (80% preference for unused personas)
- **Error handling**: Continues on errors, tracks failures

**Persona Sampling:**
- With 20M personas, we'll use ~2000 unique personas (very diverse!)
- Tracks which personas have been used to maximize diversity
- Allows some reuse (max 3 times) but prioritizes new personas
- Set `ENSURE_PERSONA_DIVERSITY = False` for pure random sampling

**Tips:**
- Generation will take **several hours** (estimate: 6-12 hours for 2000 dialogues)
- Monitor GPU memory - if you run out, reduce batch sizes in model calls
- You can stop and resume anytime - checkpoint saves automatically
- Check `generation_checkpoint.json` for current progress and persona diversity stats


## Generation Loop



In [32]:
# Configuration
NUM_DIALOGUES = 1000  # Increased for better coverage with upsampling
NUM_EU_ITEMS_PER_DIALOGUE = 4  # Increased for better diversity
NUM_EA_ITEMS_PER_DIALOGUE = 4  # Increased for better diversity
OUTPUT_DIR = OUTPUT_DIR_BASE
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "generation_checkpoint.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Output files
eu_output_path = os.path.join(OUTPUT_DIR, "eu_items.jsonl")
ea_output_path = os.path.join(OUTPUT_DIR, "ea_items.jsonl")
metadata_output_path = os.path.join(OUTPUT_DIR, "metadata.jsonl")

# Resume from checkpoint if exists
start_from = 0
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint = json.load(f)
        start_from = checkpoint.get("last_completed", 0)
        print(f"✓ Resuming from checkpoint: dialogue {start_from}")
        # Load persona tracking if available (for new checkpoints)
        if "used_persona_indices" in checkpoint:
            print(f"  - Loading persona diversity tracking from checkpoint")
else:
    print("Starting fresh generation")

# Initialize counters (load from checkpoint if resuming)
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint = json.load(f)
        qid_counter = checkpoint.get("qid_counter", 1)
        eu_count = checkpoint.get("eu_count", 0)
        ea_count = checkpoint.get("ea_count", 0)
        # Load persona tracking (may not exist in old checkpoints)
        used_persona_indices = set(checkpoint.get("used_persona_indices", []))
        print(f"  - Resuming with qid_counter: {qid_counter}")
        print(f"  - Current counts: EU={eu_count}, EA={ea_count}")
        if used_persona_indices:
            print(f"  - Unique personas used so far: {len(used_persona_indices)}")
        else:
            print(f"  - No persona tracking in checkpoint (will start fresh)")
            used_persona_indices = set()
else:
    qid_counter = 1
    eu_count = 0
    ea_count = 0
    used_persona_indices = set()

# Persona diversity settings
ENSURE_PERSONA_DIVERSITY = True  # Set to False if you want pure random sampling
MAX_REUSE_COUNT = 3  # Maximum times a persona can be reused before prioritizing unused ones

print(f"\nStarting generation of {NUM_DIALOGUES} dialogues...")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Checkpoint file: {CHECKPOINT_FILE}")
if start_from > 0:
    print(f"Progress: {start_from}/{NUM_DIALOGUES} ({start_from/NUM_DIALOGUES*100:.1f}%)")



Starting fresh generation

Starting generation of 1000 dialogues...
Output directory: /content/drive/MyDrive/685_Project_New/generated_data/mads/new_generations
Checkpoint file: /content/drive/MyDrive/685_Project_New/generated_data/mads/new_generations/generation_checkpoint.json


In [ ]:
# Main generation loop with checkpointing
# Use append mode if resuming, write mode if starting fresh
file_mode = 'a' if start_from > 0 else 'w'

with jsonlines.open(eu_output_path, mode=file_mode) as eu_writer, \
     jsonlines.open(ea_output_path, mode=file_mode) as ea_writer, \
     jsonlines.open(metadata_output_path, mode=file_mode) as meta_writer:

    # Track statistics
    failed_extractions = 0
    successful_dialogues = 0
    eu_category_counts = {cat: 0 for cat in EU_CATEGORY_CONFIG['coarse_categories'].keys()}
    ea_category_counts = {cat: 0 for cat in EA_CATEGORY_CONFIG['categories'].keys()}

    start_time = datetime.now()

    # Load persona usage count from checkpoint if resuming
    if start_from > 0 and os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            checkpoint = json.load(f)
            persona_usage_count = checkpoint.get("persona_usage_count", {})
    else:
        persona_usage_count = {}  # Track how many times each persona index is used

    for i in tqdm(range(start_from, NUM_DIALOGUES), desc="Generating dialogues", initial=start_from, total=NUM_DIALOGUES):
        # Sample persona with diversity tracking
        if ENSURE_PERSONA_DIVERSITY and len(used_persona_indices) < len(personas):
            # Prioritize unused personas, but allow some reuse
            unused_personas = [idx for idx in range(len(personas)) if idx not in used_persona_indices]
            overused_personas = [idx for idx, count in persona_usage_count.items() if count >= MAX_REUSE_COUNT]

            # If we have unused personas, prefer them (80% chance)
            if unused_personas and random.random() < 0.8:
                persona_idx = random.choice(unused_personas)
            else:
                # Sample from all, but avoid overused ones if possible
                available = [idx for idx in range(len(personas)) if idx not in overused_personas]
                if available:
                    persona_idx = random.choice(available)
                else:
                    persona_idx = random.randint(0, len(personas) - 1)
        else:
            # Pure random sampling
            persona_idx = random.randint(0, len(personas) - 1)

        persona = personas[persona_idx]
        used_persona_indices.add(persona_idx)
        persona_usage_count[persona_idx] = persona_usage_count.get(persona_idx, 0) + 1

        # Find compatible theme for this persona (avoids incoherent combinations)
        persona_text = persona.get("persona", "")
        theme = find_compatible_themes(persona_text, themes)

        # Log compatibility warnings (optional, for debugging)
        if (i + 1) % 100 == 0:  # Log every 100th dialogue
            is_compat, score, reason = check_persona_theme_compatibility(persona_text, theme)
            if not is_compat or score < 0.2:
                print(f"\n[Info] Dialogue {i+1}: Low compatibility score ({reason})")

        try:
            # Generate dialogue and extract MULTIPLE items
            eu_items, ea_items, metadata, qid_counter = generate_dialogue_and_extract(
                persona=persona,
                theme=theme,
                qid_counter=qid_counter,
                num_eu_items=NUM_EU_ITEMS_PER_DIALOGUE,
                num_ea_items=NUM_EA_ITEMS_PER_DIALOGUE
            )

            # Write all EU items
            for eu_item in eu_items:
                eu_writer.write(eu_item)
                eu_category_counts[eu_item.get('coarse_category', 'unknown')] += 1
                eu_count += 1

            # Write all EA items
            for ea_item in ea_items:
                ea_writer.write(ea_item)
                ea_category_counts[ea_item.get('category', 'unknown')] += 1
                ea_count += 1

            # Track extraction success
            if len(eu_items) > 0 or len(ea_items) > 0:
                successful_dialogues += 1
            else:
                failed_extractions += 1

            # Write metadata
            metadata["generation_id"] = i
            metadata["eu_items_extracted"] = len(eu_items)
            metadata["ea_items_extracted"] = len(ea_items)
            metadata["eu_extracted"] = len(eu_items) > 0
            metadata["ea_extracted"] = len(ea_items) > 0
            meta_writer.write(metadata)

            # Save checkpoint every 50 dialogues
            if (i + 1) % 50 == 0:
                checkpoint = {
                    "last_completed": i + 1,
                    "qid_counter": qid_counter,
                    "eu_count": eu_count,
                    "ea_count": ea_count,
                    "used_persona_indices": list(used_persona_indices),
                    "persona_usage_count": persona_usage_count,
                    "unique_personas": len(used_persona_indices),
                    "timestamp": datetime.now().isoformat()
                }
                with open(CHECKPOINT_FILE, 'w') as f:
                    json.dump(checkpoint, f, indent=2)

                # Print progress stats
                elapsed = datetime.now() - start_time
                rate = (i + 1 - start_from) / elapsed.total_seconds() if elapsed.total_seconds() > 0 else 0
                remaining = (NUM_DIALOGUES - i - 1) / rate if rate > 0 else 0
                unique_pct = (len(used_persona_indices) / (i + 1)) * 100 if (i + 1) > 0 else 0
                print(f"\n[Checkpoint {i+1}] EU: {eu_count}, EA: {ea_count} | "
                      f"Unique personas: {len(used_persona_indices)} ({unique_pct:.1f}%) | "
                      f"Rate: {rate:.2f} dialogues/sec | "
                      f"ETA: {remaining/60:.1f} min")

        except Exception as e:
            print(f"\nError in dialogue {i}: {e}")
            import traceback
            traceback.print_exc()
            failed_extractions += 1
            continue

# Final checkpoint
checkpoint = {
    "last_completed": NUM_DIALOGUES,
    "qid_counter": qid_counter,
    "eu_count": eu_count,
    "ea_count": ea_count,
    "used_persona_indices": list(used_persona_indices),
    "persona_usage_count": persona_usage_count,
    "unique_personas": len(used_persona_indices),
    "timestamp": datetime.now().isoformat(),
    "status": "complete"
}
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump(checkpoint, f, indent=2)

# Final statistics
total_time = datetime.now() - start_time
unique_persona_pct = (len(used_persona_indices) / NUM_DIALOGUES) * 100 if NUM_DIALOGUES > 0 else 0
avg_persona_reuse = sum(persona_usage_count.values()) / len(persona_usage_count) if persona_usage_count else 0

print(f"\n" + "="*60)
print(f"✓ Generation complete!")
print(f"="*60)
print(f"  - Total dialogues: {NUM_DIALOGUES}")
print(f"  - EU items generated: {eu_count}")
print(f"  - EA items generated: {ea_count}")
print(f"  - Successful dialogues: {successful_dialogues}")
print(f"  - Failed extractions: {failed_extractions}")
print(f"\n  Persona Diversity:")
print(f"  - Unique personas used: {len(used_persona_indices)} ({unique_persona_pct:.1f}% of dialogues)")
print(f"  - Average persona reuse: {avg_persona_reuse:.2f} times")
print(f"  - Max persona reuse: {max(persona_usage_count.values()) if persona_usage_count else 0} times")
print(f"\n  Performance:")
print(f"  - Total time: {total_time}")
if total_time.total_seconds() > 0:
    print(f"  - Average rate: {NUM_DIALOGUES/total_time.total_seconds():.2f} dialogues/sec")
print(f"="*60)



Generating dialogues:   5%|▍         | 47/1000 [1:14:44<24:26:29, 92.33s/it]

Generating dialogues:   5%|▌         | 50/1000 [1:20:55<29:55:33, 113.40s/it]


[Checkpoint 50] EU: 95, EA: 132 | Unique personas: 50 (100.0%) | Rate: 0.01 dialogues/sec | ETA: 1537.5 min


Generating dialogues:   6%|▋         | 63/1000 [1:42:37<26:33:46, 102.06s/it]

Generating dialogues:  10%|▉         | 99/1000 [2:42:50<22:51:14, 91.31s/it]

...


[Info] Dialogue 400: Low compatibility score (Score: 0.12 (keyword: 0.20, category: 0.00))


Generating dialogues:  40%|████      | 400/1000 [10:49:41<16:36:32, 99.65s/it] 


[Checkpoint 400] EU: 746, EA: 1150 | Unique personas: 400 (100.0%) | Rate: 0.01 dialogues/sec | ETA: 974.5 min


Generating dialogues:  40%|████      | 402/1000 [10:53:32<17:30:45, 105.43s/it]

Generating dialogues:  42%|████▏     | 415/1000 [11:13:42<16:22:08, 100.73s/it]

## Results Inspection



In [ ]:
# Load and inspect results
def inspect_results():
    eu_items = list(jsonlines.open(eu_output_path))
    ea_items = list(jsonlines.open(ea_output_path))

    print(f"EU Items: {len(eu_items)}")
    if eu_items:
        print("\nSample EU Item:")
        print(json.dumps(eu_items[0], indent=2))

    print(f"\nEA Items: {len(ea_items)}")
    if ea_items:
        print("\nSample EA Item:")
        print(json.dumps(ea_items[0], indent=2))

inspect_results()



## Notes & Next Steps

**Model Selection:**
- To use a different model, change `MODEL_NAME` in Cell 4
- Recommended models for better quality:
  - `microsoft/Phi-3-medium-4k-instruct` (14B, high quality)
  - `meta-llama/Llama-2-7b-chat-hf` (7B, requires HuggingFace auth)
  - `Qwen/Qwen2.5-7B-Instruct` (7B, good quality)
- For faster inference: `meta-llama/Llama-3.2-3B-Instruct` (3B)

**Improvements to consider:**
1. Add validation/filtering for quality control
2. Implement diversity tracking to avoid repetitive scenarios
3. Add emotion taxonomy validation
4. Scale up NUM_DIALOGUES for larger dataset generation
5. Add parallel processing for faster generation (batch inference)
6. Implement retry logic for failed extractions
7. Cache model outputs for similar prompts

**Memory Optimization:**
- 8-bit quantization enabled by default (saves ~50% memory)
- For larger models, consider 4-bit quantization or CPU offloading
- Monitor GPU memory usage and adjust batch sizes accordingly

